# Tuần 2 · Pipeline phát hiện gian lận thẻ

Tái hiện bước chấm điểm và quyết định theo chi phí trên ULB/Worldline. Notebook chạy trên CPU, với các thư viện có sẵn của Kaggle.

1. Thêm dataset [Credit Card Fraud Detection](https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud) của `mlg-ulb` qua **Add Input**.
2. Chọn **Run All**. Mặc định báo kết quả **validation** và chạy demo A/B/C. Giữ `EVALUATE_TEST = False` khi đang hoàn thiện hoặc lựa chọn cấu hình.
3. Tải `ket-qua-tuan-2.zip` ở cell cuối hoặc từ **Output** của phiên bản Kaggle đã lưu.

Phí 2,50 EUR cho mỗi cảnh báo là giả định thí nghiệm.

In [ ]:
from pathlib import Path
import hashlib
import json
import platform
import tempfile
import warnings
import zipfile

import joblib
import numpy as np
import pandas as pd
import sklearn
from IPython.display import FileLink, display
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, brier_score_loss
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

DATA_PATH = None  # Điền đường dẫn creditcard.csv nếu có nhiều tệp trùng tên.
DEMO_CSV_PATH = None  # Tùy chọn: CSV có V1–V28 và Amount.
EVALUATE_TEST = False  # Chỉ đổi sau khi đã khóa cấu hình dựa trên validation.
ALERT_FEE_CENTS = 250

KAGGLE_INPUT = Path("/kaggle/input")
WORKING_DIR = Path("/kaggle/working") if KAGGLE_INPUT.is_dir() else Path.cwd()
FEATURES = [f"V{i}" for i in range(1, 29)] + ["Amount"]
SCHEMA = ["Time", *FEATURES, "Class"]
POLICIES = ["pass_all", "alert_all", "amount_only", "A", "B", "C"]

# Mỗi lần Run All có thư mục tạm mới. File ZIP là đầu ra cần giữ.
run_directory = tempfile.TemporaryDirectory(prefix="fintech-week2-")
output_dir = Path(run_directory.name)

def write_json(filename, payload):
    (output_dir / filename).write_text(
        json.dumps(payload, indent=2, ensure_ascii=False, allow_nan=False) + "\n",
        encoding="utf-8",
    )

## 1 · Kiểm tra nguồn và tái tạo cách chia của tuần 1

Giữ dòng đầu của mỗi nhóm trùng hoàn toàn, rồi chia theo hai mốc Time đã chốt. Dấu vân tay của manifest kiểm tra cả số dòng nguồn, thứ tự và tập được gán. Time chỉ dùng để chia tập.

In [ ]:
def locate_creditcard():
    if DATA_PATH is not None:
        return Path(DATA_PATH)
    if KAGGLE_INPUT.is_dir():
        candidates = sorted(KAGGLE_INPUT.rglob("creditcard.csv"))
        if len(candidates) == 1:
            return candidates[0]
        if candidates:
            paths = "\n".join(str(path) for path in candidates)
            raise ValueError(f"Có nhiều creditcard.csv. Điền DATA_PATH ở cell cấu hình:\n{paths}")
    else:
        for root in (Path.cwd(), Path.cwd().parent):
            candidate = root / "tuan-1" / "data" / "raw" / "creditcard.csv"
            if candidate.is_file():
                return candidate
    raise FileNotFoundError("Hãy thêm dataset mlg-ulb/creditcardfraud hoặc điền DATA_PATH.")

data_path = locate_creditcard()
expected_raw_sha = "76274b691b16a6c49d3f159c883398e03ccd6d1ee12d9d8ee38f4b4b98551a89"
expected_numeric_sha = "81816cd89ef43924abf08adef46b21bd568213e06817327bfe17bdfa6d5eb4e4"
expected_manifest_sha = "3e8b0158f100c601abbd987dde73e313a29386dfd566e784243e70d865041dca"

with data_path.open("rb") as source:
    raw_sha = hashlib.file_digest(source, "sha256").hexdigest()
raw = pd.read_csv(data_path)
if list(raw.columns) != SCHEMA or raw.shape != (284807, 31):
    raise ValueError("Schema hoặc kích thước CSV khác bản ULB đã dùng ở tuần 1.")
if raw_sha != expected_raw_sha:
    # Chấp nhận khác cách đóng gói CSV nếu bảng số và thứ tự dòng giống hệt.
    round_trip = pd.read_csv(data_path, float_precision="round_trip")
    numeric_bytes = round_trip.to_numpy(dtype="<f8").tobytes(order="C")
    if hashlib.sha256(numeric_bytes).hexdigest() != expected_numeric_sha:
        raise ValueError(
            "Nội dung hoặc thứ tự dòng khác bản tuần 1. "
            "Hãy thêm đúng dataset, hoặc upload creditcard.csv từ tuan-1/data/raw."
        )
    del round_trip, numeric_bytes
if not np.isfinite(raw.to_numpy()).all() or set(raw["Class"].unique()) != {0, 1}:
    raise ValueError("Dữ liệu có giá trị không hữu hạn hoặc nhãn ngoài 0 và 1.")
if raw["Class"].sum() != 492 or (raw["Amount"] < 0).any():
    raise ValueError("Số fraud hoặc Amount không khớp kiểm tra tuần 1.")

data = raw.drop_duplicates(keep="first").copy()
data.insert(0, "source_csv_line", data.index + 2)
data = data.sort_values(["Time", "source_csv_line"], kind="stable").reset_index(drop=True)
del raw
train_end, validation_end = 120393, 145234
split_codes = np.select(
    [data["Time"] <= train_end, data["Time"] <= validation_end],
    [0, 1],
    default=2,
)
manifest_values = np.column_stack([data["source_csv_line"], data["Time"], split_codes])
manifest_sha = hashlib.sha256(manifest_values.astype("<i8").tobytes(order="C")).hexdigest()
if manifest_sha != expected_manifest_sha:
    raise ValueError("Cách chia không khớp manifest tuần 1. Kiểm tra lại CSV và pandas.")

parts = {
    name: data.loc[split_codes == code]
    for code, name in enumerate(["train", "validation", "test"])
}
split_summary = pd.DataFrame([
    {
        "split": name,
        "rows": len(part),
        "fraud": int(part["Class"].sum()),
        "time_min": int(part["Time"].min()),
        "time_max": int(part["Time"].max()),
    }
    for name, part in parts.items()
])
expected_counts = [(170236, 342), (56746, 57), (56744, 74)]
if list(split_summary[["rows", "fraud"]].itertuples(index=False, name=None)) != expected_counts:
    raise ValueError("Số giao dịch hoặc fraud từng tập khác tuần 1.")
split_summary.to_csv(output_dir / "split_summary.csv", index=False)
print(f"Dữ liệu: {data_path}")
print(f"SHA-256 CSV: {raw_sha}")
print("Đã đối chiếu manifest tuần 1.")
display(split_summary)

## 2 · Huấn luyện và khóa cấu hình trên validation

Fit bộ chuẩn hóa và Logistic Regression trên train. A dùng ngưỡng 0,5. B chọn ngưỡng xác suất giảm chi phí trên validation. C so `p × Amount` với phí cảnh báo. Amount-only chọn ngưỡng số tiền trên validation. A/B/C dùng chung một xác suất dự báo.

Chi phí quan sát = phí xử lý mọi cảnh báo + tổng Amount của fraud bị bỏ sót. Khi chọn ngưỡng, xét cả cho qua tất cả và cảnh báo tất cả. Nếu nhiều ngưỡng đồng chi phí, chọn phương án ít cảnh báo hơn.

In [ ]:
def amount_to_cents(amount):
    return np.rint(np.asarray(amount, dtype=float) * 100).astype(np.int64)

def best_threshold(values, fraud_loss_cents):
    """Trả None nếu phương án tối ưu là cho qua tất cả."""
    descending = np.argsort(-values, kind="stable")
    ranked = values[descending]
    # Chỉ xét cuối nhóm đồng điểm để mọi dòng cùng điểm nhận cùng quyết định.
    group_ends = np.flatnonzero(np.r_[ranked[:-1] != ranked[1:], True])
    alert_delta = np.cumsum(ALERT_FEE_CENTS - fraud_loss_cents[descending])
    choices = np.r_[0, alert_delta[group_ends]]
    best = int(np.argmin(choices))
    return None if best == 0 else float(ranked[group_ends[best - 1]])

train = parts["train"]
validation = parts["validation"]
model = make_pipeline(
    StandardScaler(),
    LogisticRegression(C=1.0, solver="lbfgs", max_iter=1000, random_state=42),
)
with warnings.catch_warnings():
    warnings.simplefilter("error", ConvergenceWarning)
    model.fit(train[FEATURES], train["Class"])

validation_p = model.predict_proba(validation[FEATURES])[:, 1]
validation_amount_cents = amount_to_cents(validation["Amount"])
validation_fraud_loss = validation_amount_cents * validation["Class"].to_numpy()
probability_threshold = best_threshold(validation_p, validation_fraud_loss)
amount_threshold_cents = best_threshold(validation_amount_cents, validation_fraud_loss)

config = {
    "source": "ULB / Worldline, mlg-ulb/creditcardfraud",
    "source_path": str(data_path),
    "source_csv_sha256": raw_sha,
    "week1_manifest_sha256": manifest_sha,
    "split_end_seconds": {"train": train_end, "validation": validation_end},
    "features": FEATURES,
    "preprocessing": "StandardScaler fit on train",
    "model": "LogisticRegression",
    "model_parameters": model.named_steps["logisticregression"].get_params(),
    "fit_rows": len(train),
    "alert_fee_eur": ALERT_FEE_CENTS / 100,
    "threshold_A": 0.5,
    "threshold_B": probability_threshold,
    "threshold_amount_only_eur": (
        None if amount_threshold_cents is None else amount_threshold_cents / 100
    ),
    "threshold_null_means": "pass_all",
    "threshold_tie_break": "fewer_alerts",
    "rule_C": "alert if probability times original Amount in EUR >= alert_fee_eur",
    "threshold_selection_split": "validation",
    "test_evaluated": EVALUATE_TEST,
    "versions": {
        "python": platform.python_version(),
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "scikit_learn": sklearn.__version__,
        "joblib": joblib.__version__,
    },
}
joblib.dump(model, output_dir / "model.joblib")
write_json("run_config.json", config)
print(f"Đã lưu model. Ngưỡng B: {probability_threshold!r}.")
print(f"Ngưỡng Amount-only (EUR): {config['threshold_amount_only_eur']!r}.")

## 3 · Xuất điểm và bảng so sánh

AP và Brier score đo bộ xác suất chung. Precision, recall, số cảnh báo và chi phí đo từng quy tắc. Nếu không có cảnh báo, precision được xuất dưới dạng ô trống. Kết quả test chỉ được tính nếu đã bật `EVALUATE_TEST` ở đầu notebook.

In [ ]:
def at_least(values, threshold):
    return np.zeros(len(values), dtype=bool) if threshold is None else values >= threshold

def decisions(probability, amount):
    amount = np.asarray(amount)
    return {
        "pass_all": np.zeros(len(amount), dtype=bool),
        "alert_all": np.ones(len(amount), dtype=bool),
        "amount_only": at_least(amount_to_cents(amount), amount_threshold_cents),
        "A": probability >= 0.5,
        "B": at_least(probability, probability_threshold),
        "C": probability * amount >= ALERT_FEE_CENTS / 100,
    }

def evaluate(split, frame, probability):
    labels = frame["Class"].to_numpy(dtype=bool)
    amount_cents = amount_to_cents(frame["Amount"])
    total_fraud_cents = int(amount_cents[labels].sum())
    total_fraud = int(labels.sum())
    predictions = frame[["source_csv_line", "Time", "Amount", "Class"]].copy()
    predictions["p_fraud"] = probability
    results = []
    for policy, alerts in decisions(probability, frame["Amount"]).items():
        caught = alerts & labels
        missed = ~alerts & labels
        alert_count = int(alerts.sum())
        tp = int(caught.sum())
        fp = alert_count - tp
        fn = total_fraud - tp
        missed_cents = int(amount_cents[missed].sum())
        loss_cents = ALERT_FEE_CENTS * alerts + amount_cents * missed
        cost_cents = int(loss_cents.sum())
        predictions[f"alert_{policy}"] = alerts.astype(np.int8)
        predictions[f"cost_eur_{policy}"] = loss_cents / 100
        results.append({
            "split": split,
            "policy": policy,
            "transactions": len(frame),
            "fraud": total_fraud,
            "alerts": alert_count,
            "tp": tp,
            "fp": fp,
            "fn": fn,
            "tn": len(frame) - tp - fp - fn,
            "precision": tp / alert_count if alert_count else None,
            "recall": tp / total_fraud if total_fraud else None,
            "fraud_amount_coverage": (
                (total_fraud_cents - missed_cents) / total_fraud_cents
                if total_fraud_cents else None
            ),
            "missed_fraud_amount_eur": missed_cents / 100,
            "cost_eur": cost_cents / 100,
            "cost_eur_per_1000": cost_cents * 10 / len(frame),
            "savings_vs_pass_all_eur": (total_fraud_cents - cost_cents) / 100,
        })
    predictions.to_csv(output_dir / f"predictions_{split}.csv", index=False)
    score_metrics = {
        "split": split,
        "average_precision": average_precision_score(labels, probability),
        "brier_score": brier_score_loss(labels, probability),
    }
    return pd.DataFrame(results), score_metrics

validation_metrics, validation_scores = evaluate("validation", validation, validation_p)
tables = [validation_metrics]
score_rows = [validation_scores]
if EVALUATE_TEST:
    test = parts["test"]
    test_p = model.predict_proba(test[FEATURES])[:, 1]
    test_metrics, test_scores = evaluate("test", test, test_p)
    tables.append(test_metrics)
    score_rows.append(test_scores)

policy_metrics = pd.concat(tables, ignore_index=True)
score_metrics = pd.DataFrame(score_rows)
policy_metrics.to_csv(output_dir / "policy_metrics.csv", index=False)
score_metrics.to_csv(output_dir / "score_metrics.csv", index=False)
display(score_metrics)
display(policy_metrics[[
    "split", "policy", "alerts", "precision", "recall", "fraud_amount_coverage", "cost_eur"
]])

## 4 · MVP chấm một CSV không có nhãn

Notebook mặc định tạo một nhóm nhỏ từ validation để minh họa, ưu tiên các dòng có quyết định khác nhau giữa B và C. Có thể chọn CSV riêng bằng `DEMO_CSV_PATH`. Đầu vào cần `V1`–`V28` và `Amount`. Sau khi nạp lại model từ file, notebook xuất xác suất, rủi ro tiền kỳ vọng khi cho qua và hành động A/B/C.

In [ ]:
def score_csv(csv_path, fitted_model):
    frame = pd.read_csv(csv_path)
    missing = [feature for feature in FEATURES if feature not in frame.columns]
    if missing:
        raise ValueError(f"CSV demo thiếu cột: {', '.join(missing)}")
    if frame.empty:
        raise ValueError("CSV demo cần ít nhất một dòng giao dịch.")
    features = frame[FEATURES].apply(pd.to_numeric, errors="raise")
    if not np.isfinite(features.to_numpy(dtype=float)).all() or (features["Amount"] < 0).any():
        raise ValueError("Các cột phải chứa số hữu hạn và Amount phải từ 0 trở lên.")

    probability = fitted_model.predict_proba(features)[:, 1]
    amount = features["Amount"].to_numpy()
    result = pd.DataFrame({
        "input_row": np.arange(1, len(frame) + 1),
        "amount_eur": amount,
        "p_fraud": probability,
        "expected_loss_if_pass_eur": probability * amount,
        "alert_fee_eur": ALERT_FEE_CENTS / 100,
    })
    alerts = decisions(probability, amount)
    for rule in ("A", "B", "C"):
        result[f"decision_{rule}"] = np.where(alerts[rule], "Cảnh báo", "Cho qua")
    return result

demo_input = output_dir / "demo_input.csv"
if DEMO_CSV_PATH is None:
    validation_actions = decisions(validation_p, validation["Amount"])
    demo_positions = np.unique(np.r_[
        np.arange(3),
        np.argsort(-validation_p)[:3],
        np.flatnonzero(validation_actions["B"] & ~validation_actions["C"])[:2],
        np.flatnonzero(validation_actions["C"] & ~validation_actions["B"])[:2],
    ])
    validation.iloc[demo_positions][FEATURES].to_csv(demo_input, index=False)
    demo_source = "validation rows selected for illustration"
else:
    demo_input = Path(DEMO_CSV_PATH)
    demo_source = "user CSV"

loaded_model = joblib.load(output_dir / "model.joblib")
demo_result = score_csv(demo_input, loaded_model)
if DEMO_CSV_PATH is None:
    np.testing.assert_allclose(demo_result["p_fraud"], validation_p[demo_positions], atol=1e-12)
else:
    # Giữ bản đầu vào chỉ chứa các feature mà model đã dùng.
    pd.read_csv(demo_input)[FEATURES].to_csv(output_dir / "demo_input.csv", index=False)
demo_result.to_csv(output_dir / "demo_predictions.csv", index=False)
write_json("demo_info.json", {"source": demo_source, "rows": len(demo_result)})
display(demo_result.head(15))

## 5 · Tải gói kết quả

Gói ZIP chứa bảng chia tập, cấu hình và phiên bản thư viện, model đã fit, xác suất và chi phí theo dòng, bảng metric, cùng CSV demo đầu vào và đầu ra. Tải gói này để gửi lại sau khi chạy.

In [ ]:
archive_path = WORKING_DIR / "ket-qua-tuan-2.zip"
with zipfile.ZipFile(archive_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(output_dir.iterdir()):
        archive.write(path, arcname=path.name)
print(f"Gói kết quả: {archive_path}")
link_path = archive_path.name if WORKING_DIR == Path.cwd() else str(archive_path)
display(FileLink(link_path))